In [ ]:
# =========================
# IMPORT
# =========================
import os, re
import pandas as pd
import numpy as np
import chromadb

from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler
from rank_bm25 import BM25Okapi
from collections import defaultdict

import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

# =========================
# ENV
# =========================
load_dotenv()

# =========================
# NORMALIZE ✅
# =========================
def normalize(vec):
    return vec / np.linalg.norm(vec)

# =========================
# LOAD PDF
# =========================
pdf_path = 'Data/docs_telco_v2.pdf'

md_text = pymupdf4llm.to_markdown(pdf_path, write_images=False)
md_text = re.sub(r'\n{3,}', '\n\n', md_text).strip()

# =========================
# CHUNKING
# =========================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=80,
    separators=["\n## ", "\n### ", "\n|", "\n\n"]
)

chunks = splitter.split_text(md_text)

# =========================
# DATAFRAME
# =========================
data = []
for i, chunk in enumerate(chunks):
    data.append({
        "chunk_id": f"chunk_{i}",
        "text": chunk,
        "section": chunk.split("\n")[0][:100]
    })

df = pd.DataFrame(data)

# =========================
# EMBEDDINGS (NORMALIZED ✅)
# =========================
model = SentenceTransformer("all-MiniLM-L6-v2")
df["embedding"] = df["text"].apply(lambda x: normalize(model.encode(x)))

# =========================
# BM25
# =========================
tokenized = [doc.lower().split() for doc in df["text"]]
bm25 = BM25Okapi(tokenized)

# =========================
# SPARSE VOCAB ✅
# =========================
vocab = {w:i for i, w in enumerate(set(" ".join(df["text"]).lower().split()))}

def text_to_sparse_vector(text):
    counts = defaultdict(int)
    for w in text.lower().split():
        if w in vocab:
            counts[vocab[w]] += 1

    return {
        "indices": list(counts.keys()),
        "values": list(counts.values())
    }

# =========================
# CHROMA (COSINE ✅)
# =========================
chroma = chromadb.Client()
col_name = "telco_final"

if col_name in [c.name for c in chroma.list_collections()]:
    collection = chroma.get_collection(col_name)
else:
    collection = chroma.create_collection(
        name=col_name,
        metadata={"hnsw:space": "cosine"}
    )

    collection.add(
        ids=df["chunk_id"].tolist(),
        documents=df["text"].tolist(),
        embeddings=[e.tolist() for e in df["embedding"]],
        metadatas=df[["section"]].to_dict("records")
    )

# =========================
# PINECONE (DOT PRODUCT ✅ HYBRID READY)
# =========================
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "telco-final"

if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=384,
    metric="dotproduct",  
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

index = pc.Index(index_name)

# UPSERT (DENSE + SPARSE ✅)
batch_size = 32

for i in range(0, len(df), batch_size):
    batch = []

    for j in range(i, min(i + batch_size, len(df))):
        text = df["text"].iloc[j]

        batch.append({
            "id": df["chunk_id"].iloc[j],
            "values": df["embedding"].iloc[j].tolist(),
            "sparse_values": text_to_sparse_vector(text),
            "metadata": {
                "text": text,
                "section": df["section"].iloc[j]
            }
        })

    index.upsert(vectors=batch)

# =========================
# INTENT PRUNE ✅
# =========================
def intent_prune(df, query):
    q = query.lower()

    if "registrasi" in q:
        return df[df["text"].str.contains("registrasi", case=False)]
    if "bts" in q:
        return df[df["text"].str.contains("bts", case=False)]
    if "paket" in q or "murah" in q:
        return df[df["text"].str.contains("paket|harga", case=False, regex=True)]

    return df

# =========================
# BOOST ✅
# =========================
def keyword_boost(text, query):
    text_lower = text.lower()
    q = query.lower()

    boost = 0

    if q in text_lower:
        boost += 0.3

    for w in q.split():
        if w in text_lower:
            boost += 0.1

    if "|" in text:
        boost += 0.05

    return min(boost, 0.4)

# =========================
# LOCAL HYBRID ✅
# =========================
def hybrid_search(query, top_k=5, alpha=0.6):

    candidates = intent_prune(df, query).reset_index(drop=True)
    if len(candidates) == 0:
        candidates = df.copy()

    q_emb = normalize(model.encode(query))

    res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=len(df)
    )

    dense_map = {
        cid: max(0, 1 - dist)
        for cid, dist in zip(res["ids"][0], res["distances"][0])
    }

    bm25_scores = bm25.get_scores(query.lower().split())

    scores = []

    for _, row in candidates.iterrows():
        cid = row["chunk_id"]

        dense = dense_map.get(cid, 0)
        bm = bm25_scores[df.index[df["chunk_id"] == cid][0]]

        scores.append((cid, dense, bm))

    dense_arr = np.array([s[1] for s in scores])
    bm_arr = np.array([s[2] for s in scores])

    scaler = MinMaxScaler()
    d_norm = scaler.fit_transform(dense_arr.reshape(-1,1)).flatten()
    b_norm = scaler.fit_transform(bm_arr.reshape(-1,1)).flatten()

    results = []

    for i, (cid, _, _) in enumerate(scores):
        row = candidates.iloc[i]

        base = alpha * d_norm[i] + (1 - alpha) * b_norm[i]

        if d_norm[i] < 0.15 and b_norm[i] < 0.1:
            boost = 0
        else:
            boost = keyword_boost(row["text"], query)

        results.append({
            "chunk_id": cid,
            "score": base + boost,
            "text": row["text"],
            "section": row["section"]
        })

    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]

# =========================
# PINECONE HYBRID ✅
# =========================
def pinecone_hybrid_search(query, top_k=5):

    q_dense = normalize(model.encode(query)).tolist()
    q_sparse = text_to_sparse_vector(query)

    res = index.query(
        vector=q_dense,
        sparse_vector=q_sparse,
        top_k=top_k,
        include_metadata=True
    )

    return res["matches"]

# =========================
# TEST
# =========================
queries = [
    "Berapa jumlah BTS di PT NTD",
    "Langkah registrasi prabayar",
    "Layanan apa saja di PT NTD",
    "Paket internet paling murah"
]

for q in queries:
    print("\n===================================")
    print("QUERY:", q)
    print("===================================")

    print("\n--- LOCAL HYBRID ---")
    for r in hybrid_search(q):
        print(f"\nScore: {r['score']:.4f}")
        print(r["section"])
        print(r["text"][:200])

    print("\n--- PINECONE HYBRID ---")
    for m in pinecone_hybrid_search(q):
        print(f"\nScore: {m['score']:.4f}")
        print(m["metadata"]["text"][:200])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1853.49it/s]



QUERY: Berapa jumlah BTS di PT NTD

--- LOCAL HYBRID ---

Score: 0.0000
## **4. Infrastruktur Jaringan** 
## **4. Infrastruktur Jaringan** 

NTD memiliki: 

- 45.000 BTS aktif 

- 120.000 km fiber optic backbone 

- 3 pusat Network Operation Center (NOC) 

## **Ringkasan Infrastruktur** 

## **Infrastrukt

--- PINECONE HYBRID ---

Score: 3.5312
## **1. Profil Perusahaan** 

PT Nusantara Telekomunikasi Digital (NTD) adalah perusahaan telekomunikasi nasional yang berdiri pada tahun 2008 dan berfokus pada penyediaan layanan jaringan seluler, in

Score: 3.4753
NTD menyediakan layanan 4G LTE dan telah mulai melakukan ekspansi jaringan 5G secara bertahap di kota-kota besar seperti Jakarta, Surabaya, Bandung, dan Medan. Selain layanan ritel, NTD juga memiliki 

Score: 2.3468
## **4. Infrastruktur Jaringan** 

NTD memiliki: 

- 45.000 BTS aktif 

- 120.000 km fiber optic backbone 

- 3 pusat Network Operation Center (NOC) 

## **Ringkasan Infrastruktur** 

## **Infrastrukt

Score: 1.3616
## *